# Policy Gradients from Scratch: REINFORCE on a Contextual Bandit — SOLUTIONS

**This is the solutions version: every blank is already filled in, so Run All executes end to end with no edits.** Use it to check correctness or as an answer key. The fill-in-the-blank version to work through is [`rl_golf_policy_gradient.ipynb`](https://colab.research.google.com/github/ProKil/salt-lab-rl-tutorial/blob/main/rl_golf_policy_gradient.ipynb).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ProKil/salt-lab-rl-tutorial/blob/main/rl_golf_policy_gradient_solutions.ipynb) *(CPU runtime is fine — no GPU needed)*

We derive the policy gradient formula by hand, implement the exact gradient in NumPy, verify it against a finite-difference reference, and then use it to train a policy until it finds the best action in every state. Everything happens in the simplest setting where policy gradients still have all their essential structure: a **contextual bandit** — a few states (contexts), a few actions, a reward for each state-action pair.

**The three moving parts**
- **Environment** — a contextual bandit. Draws a state $s\sim\rho$, accepts an action $a$, returns the reward $R(s,a)$. No dynamics, no episodes.
- **Policy** — a softmax linear policy $\pi_\theta(a\mid s)\propto\exp(\theta_a^\top\phi(s))$. State features in, action distribution out.
- **Algorithm** — the policy gradient: REINFORCE. Sample actions, weight their score functions by reward, ascend.

Everything you build here transfers directly to full multi-step RL by replacing the reward with the *return* $G_t$ — and it is exactly the machinery under the GRPO loop in the companion notebook [`agentic_rl_grpo_tictactoe.ipynb`](https://colab.research.google.com/github/ProKil/salt-lab-rl-tutorial/blob/main/agentic_rl_grpo_tictactoe.ipynb).

**Compute note.** This whole notebook is pure NumPy: every cell runs on CPU in seconds. No GPU, no installs beyond `numpy` (and optionally `matplotlib` for one plot).

`# ===== FILL IN THE BLANK =====` cells are filled in here. Run the test cell after each. `✓` = pass.

## Part 0 — The setup

We fix a small environment. Read this cell but you don't need to change anything.

- $S$ states, $A$ actions, feature dimension $d$.
- Each state $s$ has a feature vector $\phi(s)\in\mathbb{R}^d$; the rows of `Phi` are these vectors.
- The policy is a **softmax linear policy** with parameters $\theta\in\mathbb{R}^{A\times d}$:
$$\pi_\theta(a\mid s) = \frac{\exp\big(\theta_a^\top \phi(s)\big)}{\sum_{a'}\exp\big(\theta_{a'}^\top \phi(s)\big)}$$
- `R[s, a]` is the (expected) reward of taking action $a$ in state $s$.
- `rho[s]` is the probability of encountering state $s$.

The quantity we want to maximize is the **expected reward**
$$J(\theta) = \sum_s \rho(s)\sum_a \pi_\theta(a\mid s)\, R(s,a).$$

In [ ]:
import numpy as np

rng = np.random.default_rng(0)

S, A, d = 3, 4, 5          # states, actions, feature dimension
Phi   = rng.standard_normal((S, d))      # Phi[s] = feature vector phi(s), shape (S, d)
theta0 = 0.1 * rng.standard_normal((A, d))  # initial policy parameters, shape (A, d)
R     = rng.standard_normal((S, A))      # R[s, a] = reward, shape (S, A)
rho   = rng.random(S); rho /= rho.sum()  # state distribution, shape (S,)

print("Phi   ", Phi.shape)
print("theta0", theta0.shape)
print("R     ", R.shape)
print("rho   ", rho, "sum =", rho.sum())

## Part 1 — Derive the policy gradient (the log-derivative trick)

We want $\nabla_\theta J(\theta)$. The obstacle is that $\theta$ sits *inside* the probability we are averaging over. The **log-derivative trick** is the one idea that unlocks everything.

**Step 1 — differentiate the objective.** Since the states and rewards don't depend on $\theta$, only $\pi_\theta$ does:
$$\nabla_\theta J = \sum_s \rho(s)\sum_a \big[\nabla_\theta \pi_\theta(a\mid s)\big]\, R(s,a).$$

**Step 2 — the key identity.** For any positive function, $\nabla_\theta \pi = \pi\,\nabla_\theta \log \pi$, because $\nabla_\theta \log \pi = \frac{\nabla_\theta \pi}{\pi}$. Substituting:
$$\nabla_\theta J = \sum_s \rho(s)\sum_a \pi_\theta(a\mid s)\,\big[\nabla_\theta \log \pi_\theta(a\mid s)\big]\, R(s,a).$$

**Step 3 — read it as an expectation.** The $\rho(s)\pi_\theta(a\mid s)$ weights are exactly the probability of sampling $(s,a)$, so
$$\boxed{\;\nabla_\theta J = \mathbb{E}_{s\sim\rho,\; a\sim\pi_\theta}\big[\,R(s,a)\,\nabla_\theta \log \pi_\theta(a\mid s)\,\big]\;}$$

That boxed formula is the policy gradient. In full RL you replace $R(s,a)$ with the return $G_t$ and sum over timesteps, and the derivation is *identical* — the environment dynamics drop out for the same reason the states did here.

**Your turn (conceptual, not graded).** Two things worth working out with pen and paper before you code:
1. Why does the log-derivative trick help? (Hint: what kind of object is $\nabla_\theta\log\pi_\theta(a\mid s)$, and can we *sample* it?)
2. For our softmax linear policy, show that
$$\nabla_{\theta} \log \pi_\theta(a\mid s) = \big(\mathbf{1}_a - \pi_\theta(\cdot\mid s)\big)\,\phi(s)^\top,$$
a matrix of shape $(A, d)$, where $\mathbf{1}_a$ is the one-hot vector for the chosen action. You'll implement exactly this in Part 3.

## Part 2 — The softmax policy

First, the building blocks: a numerically stable `softmax`, and the policy probabilities $\pi_\theta(\cdot\mid s)$.

In [ ]:
def softmax(z):
    """Numerically stable softmax over the LAST axis.

    z : array, softmax computed along axis=-1
    returns : array of same shape, each slice along the last axis sums to 1.
    Hint: subtract z.max(axis=-1, keepdims=True) before exponentiating.
    """
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)


def policy_probs(theta, phi_s):
    """Policy distribution pi(.|s) for a single state.

    theta : (A, d) parameters
    phi_s : (d,)   feature vector of the state
    returns : (A,) probabilities over actions

    Hint: the action logits are theta @ phi_s, then apply softmax.
    """
    return softmax(theta @ phi_s)


def all_policy_probs(theta):
    """Vectorized helper: pi(.|s) for EVERY state at once. (Provided.)

    returns : (S, A) where row s is pi(.|s).
    """
    logits = Phi @ theta.T            # (S, A)
    return softmax(logits)

In [ ]:
# --- TEST: softmax and policy_probs (CPU) ---
def _check(name, cond):
    print(("\u2713 " if cond else "\u2717 ") + name); assert cond, name

z = np.array([[1.0, 2.0, 3.0], [0.0, 0.0, 0.0]])
sm = softmax(z)
_check("softmax rows sum to 1", np.allclose(sm.sum(axis=-1), 1.0))
_check("softmax matches reference", np.allclose(sm[1], [1/3, 1/3, 1/3]))
_check("softmax is stable on large inputs", np.all(np.isfinite(softmax(np.array([1000., 1001., 1002.])))))

p = policy_probs(theta0, Phi[0])
_check("policy_probs shape (A,)", p.shape == (A,))
_check("policy_probs is a distribution", np.isclose(p.sum(), 1.0) and np.all(p >= 0))
_check("policy_probs consistent with all_policy_probs",
       np.allclose(all_policy_probs(theta0)[0], p))
print("Part 2 passed.")

## Part 3 — Gradient of the log-policy

This is the heart of the method: $\nabla_\theta \log \pi_\theta(a\mid s)$, the *score function*. For our softmax linear policy you derived in Part 1 that it equals the outer product
$$\nabla_\theta \log \pi_\theta(a\mid s) = \big(\mathbf{1}_a - \pi_\theta(\cdot\mid s)\big)\,\phi(s)^\top \in \mathbb{R}^{A\times d}.$$

Implement it, then the test verifies your analytic gradient against a **finite-difference** approximation — the ground truth.

In [ ]:
def grad_log_pi(theta, s, a):
    """Gradient of log pi(a | s) w.r.t. theta.

    theta : (A, d)
    s     : int state index
    a     : int action index
    returns : (A, d) matrix, the gradient d log pi(a|s) / d theta.

    Hint: build the one-hot vector for action a, subtract pi(.|s),
          and take the outer product with phi(s) = Phi[s].
          np.outer(u, v) gives the (len(u), len(v)) outer product.
    """
    phi_s = Phi[s]
    p = policy_probs(theta, phi_s)          # (A,)
    onehot = np.zeros(A); onehot[a] = 1.0
    return np.outer(onehot - p, phi_s)

In [ ]:
# --- TEST: grad_log_pi vs finite differences (CPU) ---
def finite_diff(f, theta, eps=1e-6):
    """Central finite-difference gradient of scalar f(theta). (Reference tool.)"""
    g = np.zeros_like(theta)
    it = np.nditer(theta, flags=["multi_index"])
    for _ in it:
        i = it.multi_index
        tp = theta.copy(); tp[i] += eps
        tm = theta.copy(); tm[i] -= eps
        g[i] = (f(tp) - f(tm)) / (2 * eps)
    return g

ok = True
for (s, a) in [(0, 0), (1, 2), (2, 3)]:
    analytic = grad_log_pi(theta0, s, a)
    numeric  = finite_diff(lambda th: np.log(policy_probs(th, Phi[s])[a]), theta0)
    _check(f"grad_log_pi matches finite diff at (s={s}, a={a})",
           np.allclose(analytic, numeric, atol=1e-5))
    ok = ok and analytic.shape == (A, d)
_check("grad_log_pi returns shape (A, d)", ok)
print("Part 3 passed.")

## Part 4 — The exact objective and exact policy gradient

Because this environment is small, we can compute the objective $J(\theta)$ *exactly* by summing over all states and actions, and likewise the exact gradient using the boxed formula from Part 1:
$$\nabla_\theta J = \sum_s \rho(s)\sum_a \pi_\theta(a\mid s)\,R(s,a)\,\nabla_\theta \log \pi_\theta(a\mid s).$$

The test then confirms your `exact_pg` matches the finite-difference gradient of `objective` — an end-to-end check that your whole derivation is right.

In [ ]:
def objective(theta):
    """Exact expected reward J(theta) = sum_s rho(s) sum_a pi(a|s) R(s,a).

    returns : float
    Hint: all_policy_probs(theta) gives P of shape (S, A). Combine with
          rho (shape (S,)) and R (shape (S, A)) and sum everything.
    """
    P = all_policy_probs(theta)             # (S, A)
    return float(np.sum(rho[:, None] * P * R))


def exact_pg(theta):
    """Exact policy gradient of J(theta), shape (A, d).

    Sum the policy-gradient formula over all (s, a):
        grad += rho[s] * pi(a|s) * R[s, a] * grad_log_pi(theta, s, a)
    """
    grad = np.zeros_like(theta)
    P = all_policy_probs(theta)             # (S, A)
    for s in range(S):
        for a in range(A):
            grad += rho[s] * P[s, a] * R[s, a] * grad_log_pi(theta, s, a)
    return grad

In [ ]:
# --- TEST: objective and exact_pg (CPU) ---
# Reference J computed a different way (explicit python loops) to cross-check.
ref_J = sum(rho[s] * all_policy_probs(theta0)[s, a] * R[s, a]
            for s in range(S) for a in range(A))
_check("objective matches reference value", np.isclose(objective(theta0), ref_J))

analytic_g = exact_pg(theta0)
numeric_g  = finite_diff(objective, theta0)
_check("exact_pg shape (A, d)", analytic_g.shape == (A, d))
_check("exact_pg matches finite diff of J",
       np.allclose(analytic_g, numeric_g, atol=1e-5))
print(f"max abs error vs finite diff: {np.abs(analytic_g - numeric_g).max():.2e}")
print("Part 4 passed.")

## Part 5 — The REINFORCE (Monte Carlo) estimator

In a real problem you can't enumerate every $(s,a)$ — you only get *samples*. The boxed formula is an expectation, so we estimate it by sampling: draw $s\sim\rho$, then $a\sim\pi_\theta(\cdot\mid s)$, observe reward $R(s,a)$, and average
$$\hat g = \frac{1}{N}\sum_{i=1}^{N} R(s_i, a_i)\,\nabla_\theta \log \pi_\theta(a_i\mid s_i).$$
This is **REINFORCE**. It's unbiased, so with enough samples $\hat g \to \nabla_\theta J$. The test checks that your estimate lands close to the *exact* gradient from Part 4.

In [ ]:
def reinforce_estimate(theta, N, seed=0):
    """Monte Carlo (REINFORCE) estimate of the policy gradient.

    Draw N samples: s ~ rho, a ~ pi(.|s). Return the average of
        R[s, a] * grad_log_pi(theta, s, a)
    returns : (A, d)

    Hint: use r = np.random.default_rng(seed); r.choice(S, p=rho) for the state,
          and r.choice(A, p=policy_probs(theta, Phi[s])) for the action.
    """
    r = np.random.default_rng(seed)
    grad = np.zeros_like(theta)
    for _ in range(N):
        s = r.choice(S, p=rho)
        a = r.choice(A, p=policy_probs(theta, Phi[s]))
        grad += R[s, a] * grad_log_pi(theta, s, a)
    return grad / N

In [ ]:
# --- TEST: REINFORCE estimate approaches the exact gradient (CPU) ---
exact_g = exact_pg(theta0)
est_g   = reinforce_estimate(theta0, N=100_000, seed=0)
err = np.abs(est_g - exact_g).max()
print(f"max abs error (N=100k): {err:.4f}")
_check("REINFORCE estimate close to exact gradient", err < 0.02)
_check("estimate points the same way as exact gradient",
       np.dot(est_g.ravel(), exact_g.ravel()) > 0)
print("Part 5 passed. (It's a stochastic estimate, so it won't be exact.)")

## Part 6 — Putting it together: train the policy

Now use the gradient to actually *improve* the policy with gradient **ascent**: repeatedly step $\theta \leftarrow \theta + \alpha\,\nabla_\theta J$. If everything above is correct, $J$ should climb toward the best achievable value $\sum_s \rho(s)\max_a R(s,a)$.

In [ ]:
theta = theta0.copy()
alpha = 0.5
history = [objective(theta)]

for step in range(200):
    g = exact_pg(theta)                 # try reinforce_estimate(theta, 2000, seed=step) instead!
    # One gradient-ASCENT step on theta using learning rate alpha and gradient g.
    theta = theta + alpha * g
    history.append(objective(theta))

J_best = float(np.sum(rho * R.max(axis=1)))
print(f"J start : {history[0]:.4f}")
print(f"J end   : {history[-1]:.4f}")
print(f"J best  : {J_best:.4f}  (upper bound)")

_check("policy improved", history[-1] > history[0] + 0.1)
_check("reached near-optimal J", history[-1] > 0.95 * J_best)
print("Part 6 passed \u2014 you trained a policy with your own gradient!")

In [ ]:
# Optional: visualize the learning curve (requires matplotlib)
try:
    import matplotlib.pyplot as plt
    plt.plot(history, label="J(theta)")
    plt.axhline(J_best, ls="--", color="gray", label="upper bound")
    plt.xlabel("gradient ascent step"); plt.ylabel("expected reward J")
    plt.title("Policy improves under gradient ascent"); plt.legend(); plt.show()
except ImportError:
    print("matplotlib not installed \u2014 skipping the plot.")

## Where to go next

- **Full RL.** Replace the immediate reward $R(s,a)$ with the *return* $G_t=\sum_{k\ge t}\gamma^{k-t} r_k$ and sum the score over a whole trajectory. The derivation and code structure are unchanged.
- **Baselines / variance reduction.** Subtract a baseline $b(s)$ from the return: $\nabla_\theta J = \mathbb{E}[(G_t - b(s))\nabla_\theta\log\pi]$. It stays unbiased (since $\mathbb{E}[\nabla_\theta\log\pi]=0$) but has lower variance. This leads to actor-critic and advantage estimation.
- **Agentic RL.** The companion notebook [`agentic_rl_grpo_tictactoe.ipynb`](https://colab.research.google.com/github/ProKil/salt-lab-rl-tutorial/blob/main/agentic_rl_grpo_tictactoe.ipynb) is this exact loop at scale: the softmax linear policy becomes a small LLM, the bandit becomes tic-tac-toe, and the reward-weighted score becomes GRPO's group-relative advantage.
- **Continuous actions.** Swap the softmax for a Gaussian policy; the score function $\nabla_\theta\log\pi$ changes but the boxed formula does not.

---
## Appendix — Solution key

Try the exercises yourself first. If you're stuck, run this cell to load reference implementations, then re-run the test cells above.

In [ ]:
# ================= SOLUTION KEY (peek only if stuck) =================
def softmax(z):
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)

def policy_probs(theta, phi_s):
    return softmax(theta @ phi_s)

def grad_log_pi(theta, s, a):
    phi_s = Phi[s]
    p = policy_probs(theta, phi_s)
    onehot = np.zeros(A); onehot[a] = 1.0
    return np.outer(onehot - p, phi_s)

def objective(theta):
    P = all_policy_probs(theta)
    return float(np.sum(rho[:, None] * P * R))

def exact_pg(theta):
    grad = np.zeros_like(theta)
    P = all_policy_probs(theta)
    for s in range(S):
        for a in range(A):
            grad += rho[s] * P[s, a] * R[s, a] * grad_log_pi(theta, s, a)
    return grad

def reinforce_estimate(theta, N, seed=0):
    r = np.random.default_rng(seed)
    grad = np.zeros_like(theta)
    for _ in range(N):
        s = r.choice(S, p=rho)
        a = r.choice(A, p=policy_probs(theta, Phi[s]))
        grad += R[s, a] * grad_log_pi(theta, s, a)
    return grad / N

print("Reference solutions loaded. Re-run the test cells to verify.")
# The training-loop blank is simply:  theta = theta + alpha * g